In [1]:
%pip install -U langchain langchain-groq langchain-core streamlit

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain (from versions: none)
ERROR: No matching distribution found for langchain


In [2]:
import os

if os.getenv("GROQ_API_KEY"):
    print("Groq API key found successfully!")
else:
    print("API key not found. Please set GROQ_API_KEY in Windows.")

API key not found. Please set GROQ_API_KEY in Windows.


In [3]:
import ast
import operator as op

from langchain_core.tools import tool

allowed_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.UAdd: op.pos
}


def calculate_expression(node):
    if isinstance(node, ast.Constant) and isinstance(
        node.value, (int, float)
    ):
        return node.value

    elif isinstance(node, ast.BinOp):
        operator_type = type(node.op)

        if operator_type in allowed_operators:
            left = calculate_expression(node.left)
            right = calculate_expression(node.right)

            return allowed_operators[operator_type](left, right)

    elif isinstance(node, ast.UnaryOp):
        operator_type = type(node.op)

        if operator_type in allowed_operators:
            return allowed_operators[operator_type](
                calculate_expression(node.operand)
            )

    raise ValueError("Invalid mathematical expression")


@tool
def calculator(expression: str) -> str:
    """Calculate arithmetic expressions such as 10 * 5 or 100 / 4."""
    try:
        tree = ast.parse(expression, mode="eval")
        answer = calculate_expression(tree.body)
        return str(answer)

    except Exception as error:
        return f"Calculation error: {error}"


print(calculator.invoke({"expression": "10 * 5"}))

50


In [5]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("Groq model connected successfully!")

Groq model connected successfully!


In [7]:
%pip install -U langchain langchain-core langchain-groq

  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_core-1.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.16-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.20-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.3.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached langgraph_sdk-0.4.9-py3-none-any.whl.metadata (5.1 kB)
  Using cached ormsgpack-1.13.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
Using cached langchain-1.4.7-py3-none-any.whl (170 kB)
Using cached langchain_core-1.7.0-py3-none-any.whl (574 kB)
Using cached langgraph-1.2.16-py3-none-any.whl (260 kB)
Using cached langgraph_checkpoint-4.3.0-py3-none-any.whl (58 kB)
Using cached langgraph_prebuilt-1.1.1-py3-none-any.whl (42 kB)
Using cached langgraph_sdk-0.4.9-py3-none-any.whl (167 kB)

  Attempting uninstall: langchain-protocol

    Found existing 

In [5]:
import ast
import operator as op

from langchain_core.tools import tool

# Allowed mathematical operators
allowed_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
    ast.UAdd: op.pos
}


# Function to calculate mathematical expressions
def calculate_expression(node):

    # Check numbers
    if isinstance(node, ast.Constant) and isinstance(
        node.value, (int, float)
    ):
        return node.value

    # Calculate addition, subtraction, multiplication, division
    if isinstance(node, ast.BinOp):
        operator_type = type(node.op)

        if operator_type in allowed_operators:
            left = calculate_expression(node.left)
            right = calculate_expression(node.right)

            return allowed_operators[operator_type](left, right)

    # Handle positive and negative numbers
    if isinstance(node, ast.UnaryOp):
        operator_type = type(node.op)

        if operator_type in allowed_operators:
            return allowed_operators[operator_type](
                calculate_expression(node.operand)
            )

    raise ValueError("Invalid mathematical expression")


# Create calculator tool
@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions such as 10 * 5 or 100 / 4."""

    try:
        tree = ast.parse(expression, mode="eval")
        answer = calculate_expression(tree.body)

        return str(answer)

    except Exception as error:
        return f"Calculation error: {error}"


# Test calculator
print(calculator.invoke({"expression": "10 * 5"}))

50


In [7]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Get the API key if it is not already available
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

# Create the LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("Groq model connected successfully!")

Groq model connected successfully!


In [9]:
from langchain.agents import create_agent

tools = [calculator]

math_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful math tutor. "
        "Understand word problems and explain solutions step by step. "
        "Use the calculator tool for arithmetic calculations. "
        "Give a clear final answer. "
        "For simple algebra, explain the steps used to solve the equation."
    )
)

print("Text-to-Math Agent created successfully!")
response = math_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 25 multiplied by 4?"}
    ]
})

print(response["messages"][-1].content)

Text-to-Math Agent created successfully!
25 × 4 = 100.


In [10]:
test_questions = [
    "What is 150 + 250?",
    "What is 20% of 500?",
    "A notebook costs 40 rupees. What is the cost of 6 notebooks?"
]

for question in test_questions:
    response = math_agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    print("Question:", question)
    print("Answer:", response["messages"][-1].content)
    print("-" * 50)

Question: What is 150 + 250?
Answer: The sum of 150 and 250 is **400**.
--------------------------------------------------
Question: What is 20% of 500?
Answer: To find 20 % of 500, multiply the number by the percentage expressed as a decimal:

\[
20\% = \frac{20}{100} = 0.20
\]

\[
0.20 \times 500 = 100
\]

So, **20 % of 500 is 100**.
--------------------------------------------------
Question: A notebook costs 40 rupees. What is the cost of 6 notebooks?
Answer: Each notebook costs ₹40.  
To find the total cost for 6 notebooks, multiply the price of one notebook by the number of notebooks:

\[
6 \times 40 = 240
\]

So, 6 notebooks cost **₹240**.
--------------------------------------------------
